# Selective Prediction

Reject uncertain predictions to improve reliability.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from deepuq.models import MLP
from deepuq.methods.selective import SelectivePredictor
from deepuq.methods.mc_dropout import MCDropoutWrapper
from deepuq.metrics import coverage, selective_mse, aurc

## Setup

In [ ]:
np.random.seed(42)

# Generate data with OOD region (gap between -1 and 1)
x_train_left = np.random.uniform(-3, -1, 50).astype(np.float32)
x_train_right = np.random.uniform(1, 3, 50).astype(np.float32)
x_train = np.concatenate([x_train_left, x_train_right])
y_train = (np.sin(x_train) + np.random.normal(0, 0.1, 100)).astype(np.float32)
x_test = np.linspace(-5, 5, 200).astype(np.float32)
y_test = np.sin(x_test).astype(np.float32)

x_train_t = torch.from_numpy(x_train).unsqueeze(-1)
y_train_t = torch.from_numpy(y_train).unsqueeze(-1)
x_test_t = torch.from_numpy(x_test).unsqueeze(-1)
y_test_t = torch.from_numpy(y_test).unsqueeze(-1)

# Train MLP with dropout
model = MLP(1, [64, 64], 1, dropout_rate=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = torch.nn.MSELoss()

for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(x_train_t), y_train_t)
    loss.backward()
    optimizer.step()

print(f"Final training loss: {loss.item():.4f}")

# Wrap with MC Dropout
mc_model = MCDropoutWrapper(model, n_samples=50)

## Selective Prediction

In [ ]:
# Create selective predictor
selector = SelectivePredictor(mc_model)

# Find threshold for 80% coverage
threshold = selector.find_threshold(x_test_t, target_coverage=0.8)
print(f"Threshold for 80% coverage: {threshold:.4f}")

# Predict with rejection
mean, std, mask = selector.predict_with_rejection(x_test_t, threshold=threshold)
print(f"Accepted: {mask.sum()}/{len(mask)} predictions ({100*mask.float().mean():.1f}%)")

## Evaluate

In [ ]:
# Evaluate selective prediction
metrics = selector.evaluate(x_test_t, y_test_t, threshold=threshold)
print(f"Coverage: {metrics['coverage']:.3f}")
print(f"Selective MSE: {metrics['selective_mse']:.4f}")
print(f"AURC: {metrics['aurc']:.4f}")

## Risk-Coverage Tradeoff

In [ ]:
# Plot risk vs coverage at different thresholds
thresholds = np.linspace(0.01, std.numpy().max(), 50)
coverages = []
risks = []

mean_np = mean.numpy()
std_np = std.numpy()

for t in thresholds:
    accepted = std_np <= t
    cov = accepted.mean()
    if cov > 0:
        risk = ((mean_np[accepted] - y_test[accepted]) ** 2).mean()
    else:
        risk = np.nan
    coverages.append(cov)
    risks.append(risk)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Risk-coverage curve
axes[0].plot(coverages, risks, "b-", linewidth=2)
axes[0].axvline(x=0.8, color="r", linestyle="--", alpha=0.5, label="80% coverage")
axes[0].set_xlabel("Coverage")
axes[0].set_ylabel("Selective Risk (MSE)")
axes[0].set_title("Risk-Coverage Curve")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Spatial visualization
axes[1].scatter(x_train, y_train, alpha=0.3, label="Train data", color="gray")
axes[1].plot(x_test, np.sin(x_test), "k--", alpha=0.5, label="True function")

accepted_mask = mask.numpy()
axes[1].scatter(x_test[accepted_mask], mean_np[accepted_mask],
               color="green", s=10, label="Accepted")
axes[1].scatter(x_test[~accepted_mask], mean_np[~accepted_mask],
               color="red", s=10, label="Rejected")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_title("Selective Predictions (green=accepted, red=rejected)")
axes[1].legend()

plt.tight_layout()
plt.show()